1. Read the individual variant data and display the first 3 rows of each table.
2. Count the number of customers known to be female.
3. Display 5 reviews with purchase id > 100.
4. Find the average purchase value for each customer_id, display the first 7 rows.
5. Find the names of customers (no more than 20) with purchase value < 10.

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Load CSV").getOrCreate()

In [4]:
df_customers = spark.read.csv('customers.csv',header=True, inferSchema=True)
df_reviews = spark.read.csv('purchase_reviews.csv',header=True, inferSchema=True)
df_purchases = spark.read.csv('purchases.csv',header=True, inferSchema=True)

In [7]:
df_customers.show(3)
df_reviews.show(3)
df_purchases.show(3)

+---+---------+--------+------+----------+-------------+------+------+-----------+
| id|firstname|lastname|gender|  birthday|    interests|haspet|hascar|haschildren|
+---+---------+--------+------+----------+-------------+------+------+-----------+
|  1|   Daniel|   Brown|  NULL|      NULL|Cars, Reading|  NULL|  NULL|       NULL|
|  2|  Natalie|  Hughes|Female|1990-03-28|  Video Games|  NULL|   Yes|       NULL|
|  3|     Ruth|  Bennet|  NULL|1988-09-26|         Cats|   Yes|  NULL|         No|
+---+---------+--------+------+----------+-------------+------+------+-----------+
only showing top 3 rows
+------+--------------------+---+-----------+
|rating|             comment| id|purchase_id|
+------+--------------------+---+-----------+
|   5.0|Fantastic deal! W...|  1|       3733|
|   5.0|Fantastic service...|  2|       1300|
|  NULL|Nice item, just w...|  3|       1036|
+------+--------------------+---+-----------+
only showing top 3 rows
+---+-----------+-------------------+------+-----

In [9]:
count_fem_customers = df_customers.filter(df_customers.gender == "Female").count()
count_fem_customers

260

In [24]:
filtered_df_reviews = df_reviews.filter(df_reviews.purchase_id > 100)
filtered_df_reviews.show(5)

+------+--------------------+---+-----------+
|rating|             comment| id|purchase_id|
+------+--------------------+---+-----------+
|   5.0|Fantastic deal! W...|  1|       3733|
|   5.0|Fantastic service...|  2|       1300|
|  NULL|Nice item, just w...|  3|       1036|
|   4.0|Good product, a b...|  4|       4355|
|   1.0|Terrible experien...|  5|       3879|
+------+--------------------+---+-----------+
only showing top 5 rows


In [17]:
mean_cost = df_purchases.filter(df_purchases.cancelled.isNull() | df_purchases.cancelled == "No").groupBy("customer_id").mean("cost")
mean_cost.show(7)

+-----------+------------------+
|customer_id|         avg(cost)|
+-----------+------------------+
|        463|            621.17|
|        148|477.59666666666664|
|        496|            633.37|
|        833|            502.45|
|        471|213.74666666666667|
|        737|           952.205|
|        897|          527.3525|
+-----------+------------------+
only showing top 7 rows


In [23]:
cost_val_10 = df_purchases.filter(df_purchases.cost < 10).join(df_customers, df_purchases.customer_id == df_customers.id).select("firstname", "lastname").distinct().limit(20)
cost_val_10.show()

+---------+--------+
|firstname|lastname|
+---------+--------+
|   Janice| Roberts|
|Christina|   Baker|
|  Timothy|   Clark|
|    Terry| Mendoza|
|    Helen|  Bennet|
|  Gabriel|  Wilson|
| Isabella|Mitchell|
|  Bradley|   Myers|
|   Rachel|   Ortiz|
|   Thomas|   Allen|
|     Juan|   Ortiz|
|  Gregory|  Carter|
|     Lori|  Thomas|
|     Anna|  Morris|
|Charlotte|    Diaz|
|  Heather|   Davis|
|   George|   Evans|
|   Janice|Phillips|
| Nicholas| Sanchez|
|     Liam|Mitchell|
+---------+--------+

